# 🏏 IPL Match Winner Prediction
### Using Machine Learning Algorithms

**Dataset:** IPL matches from 2007/08 to 2024

**Two Prediction Modes:**
1. **Pre-Match Prediction** – predict winner before the game using team history, toss, venue
2. **Live In-Match Prediction** – predict winner during 2nd innings using runs, balls, wickets, overs

**Models Used:** Logistic Regression | Random Forest | SVM | Naive Bayes

## Step 1 – Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

print("All libraries imported successfully ✅")

## Step 2 – Load the Dataset

In [ ]:
matches    = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

print("Matches shape:", matches.shape)
print("Deliveries shape:", deliveries.shape)
matches.head()

In [ ]:
deliveries.head()

## Step 3 – Exploratory Data Analysis

In [ ]:
print("Missing values in matches:")
print(matches.isnull().sum())

In [ ]:
print("Seasons covered:", sorted(matches['season'].unique()))
print("Total matches:", len(matches))

In [ ]:
team_matches = matches['team1'].value_counts() + matches['team2'].value_counts()
team_matches = team_matches.fillna(0).sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 5))
team_matches.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Top 10 Teams by Matches Played', fontsize=14)
plt.xlabel('Team')
plt.ylabel('Number of Matches')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
top_winners = matches['winner'].value_counts().head(10)

plt.figure(figsize=(12, 5))
top_winners.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Top 10 Teams by Total Wins', fontsize=14)
plt.xlabel('Team')
plt.ylabel('Number of Wins')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
toss_decision_counts = matches['toss_decision'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(toss_decision_counts, labels=toss_decision_counts.index,
        autopct='%1.1f%%', startangle=90, colors=['#66b3ff','#ff9999'])
plt.title('Toss Decision Distribution')
plt.show()

In [ ]:
matches_clean = matches.dropna(subset=['winner'])
toss_win = (matches_clean['toss_winner'] == matches_clean['winner']).mean()
print(f"% of times toss winner also won the match: {toss_win*100:.1f}%")

---
# PART A – Pre-Match Prediction
Predict the winner before the match starts using team stats, toss, venue and head-to-head records.

## Step 4 – Pre-Match Feature Engineering

In [ ]:
matches = matches.dropna(subset=['winner'])
matches = matches.sort_values('date').reset_index(drop=True)
matches['city'] = matches['city'].fillna('Unknown')

print("Matches after cleaning:", matches.shape)

In [ ]:
matches['team1_win']             = (matches['winner'] == matches['team1']).astype(int)
matches['toss_decision_encoded'] = (matches['toss_decision'] == 'bat').astype(int)
matches['toss_is_team1']         = (matches['toss_winner'] == matches['team1']).astype(int)

seasons_map = {s: i for i, s in enumerate(sorted(matches['season'].unique()))}
matches['season_enc'] = matches['season'].map(seasons_map)

In [ ]:
team_wins     = matches['winner'].value_counts()
team_total    = matches['team1'].value_counts() + matches['team2'].value_counts()
team_win_rate = (team_wins / team_total).fillna(0.5)

matches['team1_win_rate'] = matches['team1'].map(team_win_rate).fillna(0.5)
matches['team2_win_rate'] = matches['team2'].map(team_win_rate).fillna(0.5)
matches['win_rate_diff']  = matches['team1_win_rate'] - matches['team2_win_rate']

print("Team win rates:")
print(team_win_rate.sort_values(ascending=False).head(8))

In [ ]:
h2h_dict = {}
for _, row in matches.iterrows():
    t1, t2, w = row['team1'], row['team2'], row['winner']
    key = tuple(sorted([t1, t2]))
    if key not in h2h_dict:
        h2h_dict[key] = {'total': 0}
    h2h_dict[key]['total'] += 1
    h2h_dict[key][w] = h2h_dict[key].get(w, 0) + 1

h2h_rates = []
for _, row in matches.iterrows():
    t1, t2 = row['team1'], row['team2']
    key    = tuple(sorted([t1, t2]))
    info   = h2h_dict.get(key, {})
    total  = info.get('total', 0)
    h2h_rates.append(info.get(t1, 0) / total if total > 0 else 0.5)

matches['h2h_win_rate'] = h2h_rates
print("Head-to-head win rates computed ✅")

In [ ]:
venue_stats = matches.groupby(['team1', 'venue'])['team1_win'].mean().reset_index()
venue_stats.columns = ['team1', 'venue', 'venue_win_rate']
matches = matches.merge(venue_stats, on=['team1', 'venue'], how='left')
matches['venue_win_rate'] = matches['venue_win_rate'].fillna(0.5)

print("Venue win rates computed ✅")

In [ ]:
le_team = LabelEncoder()
all_teams = pd.concat([matches['team1'], matches['team2']]).unique()
le_team.fit(all_teams)

matches['team1_enc']       = le_team.transform(matches['team1'])
matches['team2_enc']       = le_team.transform(matches['team2'])
matches['toss_winner_enc'] = le_team.transform(matches['toss_winner'])

le_city = LabelEncoder()
matches['city_enc'] = le_city.fit_transform(matches['city'])

print("Label encoding done ✅")

In [ ]:
pre_features = [
    'team1_enc', 'team2_enc', 'toss_winner_enc',
    'toss_decision_encoded', 'city_enc', 'season_enc',
    'team1_win_rate', 'team2_win_rate', 'win_rate_diff',
    'toss_is_team1', 'h2h_win_rate', 'venue_win_rate'
]

X_pre = matches[pre_features]
y_pre = matches['team1_win']

print("Pre-match feature matrix shape:", X_pre.shape)

## Step 5 – Train-Test Split (Pre-Match)

In [ ]:
X_pre_train, X_pre_test, y_pre_train, y_pre_test = train_test_split(
    X_pre, y_pre, test_size=0.2, random_state=42, stratify=y_pre
)

print("Training size:", X_pre_train.shape)
print("Testing size: ", X_pre_test.shape)

## Step 6 – Pre-Match Model Training

### 6.1 – Logistic Regression

In [ ]:
lr_pre = LogisticRegression(max_iter=2000, C=10)
lr_pre.fit(X_pre_train, y_pre_train)

lr_pre_train_acc = accuracy_score(y_pre_train, lr_pre.predict(X_pre_train))
lr_pre_test_acc  = accuracy_score(y_pre_test,  lr_pre.predict(X_pre_test))

print(f"Logistic Regression -> Train Accuracy: {lr_pre_train_acc*100:.2f}%")
print(f"Logistic Regression -> Test Accuracy : {lr_pre_test_acc*100:.2f}%")

### 6.2 – Random Forest

In [ ]:
rf_pre = RandomForestClassifier(n_estimators=300, max_depth=12, random_state=42)
rf_pre.fit(X_pre_train, y_pre_train)

rf_pre_train_acc = accuracy_score(y_pre_train, rf_pre.predict(X_pre_train))
rf_pre_test_acc  = accuracy_score(y_pre_test,  rf_pre.predict(X_pre_test))

print(f"Random Forest -> Train Accuracy: {rf_pre_train_acc*100:.2f}%")
print(f"Random Forest -> Test Accuracy : {rf_pre_test_acc*100:.2f}%")

### 6.3 – SVM

In [ ]:
svm_pre = SVC(kernel='rbf', C=10, probability=True, random_state=42)
svm_pre.fit(X_pre_train, y_pre_train)

svm_pre_train_acc = accuracy_score(y_pre_train, svm_pre.predict(X_pre_train))
svm_pre_test_acc  = accuracy_score(y_pre_test,  svm_pre.predict(X_pre_test))

print(f"SVM -> Train Accuracy: {svm_pre_train_acc*100:.2f}%")
print(f"SVM -> Test Accuracy : {svm_pre_test_acc*100:.2f}%")

### 6.4 – Naive Bayes

In [ ]:
nb_pre = GaussianNB()
nb_pre.fit(X_pre_train, y_pre_train)

nb_pre_train_acc = accuracy_score(y_pre_train, nb_pre.predict(X_pre_train))
nb_pre_test_acc  = accuracy_score(y_pre_test,  nb_pre.predict(X_pre_test))

print(f"Naive Bayes -> Train Accuracy: {nb_pre_train_acc*100:.2f}%")
print(f"Naive Bayes -> Test Accuracy : {nb_pre_test_acc*100:.2f}%")

## Step 7 – Pre-Match Model Comparison

In [ ]:
pre_results_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'SVM', 'Naive Bayes'],
    'Train Accuracy (%)': [
        round(lr_pre_train_acc*100, 2), round(rf_pre_train_acc*100, 2),
        round(svm_pre_train_acc*100, 2), round(nb_pre_train_acc*100, 2)
    ],
    'Test Accuracy (%)': [
        round(lr_pre_test_acc*100, 2), round(rf_pre_test_acc*100, 2),
        round(svm_pre_test_acc*100, 2), round(nb_pre_test_acc*100, 2)
    ]
})
pre_results_df = pre_results_df.sort_values('Test Accuracy (%)', ascending=False).reset_index(drop=True)
print("Pre-Match Model Results:")
pre_results_df

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x     = np.arange(len(pre_results_df))
width = 0.35

bars1 = ax.bar(x - width/2, pre_results_df['Train Accuracy (%)'], width, label='Train Accuracy', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, pre_results_df['Test Accuracy (%)'],  width, label='Test Accuracy',  color='coral',     edgecolor='black')

ax.set_xlabel('Model')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Pre-Match Model Accuracy Comparison', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(pre_results_df['Model'], rotation=15, ha='right')
ax.set_ylim(0, 115)
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f"{bar.get_height():.1f}", ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f"{bar.get_height():.1f}", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
pre_model_dict = {
    'Logistic Regression': (lr_pre,  lr_pre_test_acc),
    'Random Forest':       (rf_pre,  rf_pre_test_acc),
    'SVM':                 (svm_pre, svm_pre_test_acc),
    'Naive Bayes':         (nb_pre,  nb_pre_test_acc)
}

best_pre_name  = max(pre_model_dict, key=lambda k: pre_model_dict[k][1])
best_pre_model, best_pre_acc = pre_model_dict[best_pre_name]

print("=" * 50)
print(f"  🏆 Best Pre-Match Model  : {best_pre_name}")
print(f"  ✅ Test Accuracy         : {best_pre_acc*100:.2f}%")
print("=" * 50)

---
# PART B – Live In-Match Prediction
Predict the winner **during the 2nd innings** using real match inputs:
- **Runs scored** so far by batting team
- **Balls bowled** (or current over)
- **Wickets fallen**
- **Runs left** to win, **balls left**, **CRR**, **RRR**

## Step 8 – Build Live In-Match Dataset from Deliveries

In [ ]:
deliveries['current_score']  = deliveries.groupby(['match_id', 'inning'])['total_runs'].cumsum()
deliveries['balls_bowled']   = deliveries.groupby(['match_id', 'inning']).cumcount() + 1
deliveries['wickets_fallen'] = deliveries.groupby(['match_id', 'inning'])['is_wicket'].cumsum()

print("Ball-by-ball stats computed ✅")
deliveries[['match_id','batting_team','bowling_team','current_score','balls_bowled','wickets_fallen']].head(8)

In [ ]:
first_innings = deliveries[deliveries['inning'] == 1]
target_scores = first_innings.groupby('match_id')['total_runs'].sum().reset_index()
target_scores.columns = ['match_id', 'target']
target_scores['target'] += 1

print("Target scores computed ✅")
print(target_scores.head())

In [ ]:
second_innings = deliveries[deliveries['inning'] == 2].copy()
second_innings = second_innings.merge(target_scores, on='match_id')

second_innings['runs_left']    = second_innings['target'] - second_innings['current_score']
second_innings['balls_left']   = 120 - second_innings['balls_bowled']
second_innings['wickets_left'] = 10  - second_innings['wickets_fallen']
second_innings['current_over'] = ((second_innings['balls_bowled'] - 1) // 6) + 1
second_innings['crr']          = (second_innings['current_score'] / second_innings['balls_bowled']) * 6
second_innings['rrr']          = (second_innings['runs_left'] / second_innings['balls_left'].replace(0, 1)) * 6

winner_map = matches.set_index('id')['winner'].to_dict()
second_innings['match_winner'] = second_innings['match_id'].map(winner_map)
second_innings['batting_win']  = (second_innings['batting_team'] == second_innings['match_winner']).astype(int)

print("2nd innings features computed ✅")
second_innings[['batting_team','bowling_team','current_score','balls_bowled',
                'wickets_fallen','runs_left','balls_left','crr','rrr','batting_win']].head(5)

In [ ]:
live_df = second_innings[[
    'batting_team', 'bowling_team',
    'current_score', 'runs_left', 'balls_left', 'wickets_left',
    'current_over', 'target', 'crr', 'rrr', 'batting_win'
]].dropna()

live_df = live_df[live_df['balls_left'] > 0].sample(frac=1, random_state=42).reset_index(drop=True)

print("Live dataset shape:", live_df.shape)
print("Target distribution:\n", live_df['batting_win'].value_counts())

## Step 9 – Encode Teams for Live Model

In [ ]:
le_live = LabelEncoder()
all_live_teams = pd.concat([live_df['batting_team'], live_df['bowling_team']]).unique()
le_live.fit(all_live_teams)

live_df['bat_enc']  = le_live.transform(live_df['batting_team'])
live_df['bowl_enc'] = le_live.transform(live_df['bowling_team'])

live_features = [
    'bat_enc', 'bowl_enc',
    'current_score', 'runs_left', 'balls_left', 'wickets_left',
    'current_over', 'target', 'crr', 'rrr'
]

X_live = live_df[live_features]
y_live = live_df['batting_win']

print("Live feature matrix shape:", X_live.shape)
print("Features:", live_features)

## Step 10 – Train-Test Split (Live)

In [ ]:
X_live_train, X_live_test, y_live_train, y_live_test = train_test_split(
    X_live, y_live, test_size=0.2, random_state=42
)

print("Live Train size:", X_live_train.shape)
print("Live Test size: ", X_live_test.shape)

## Step 11 – Live Model Training

### 11.1 – Logistic Regression (Live)

In [ ]:
lr_live = LogisticRegression(max_iter=1000, C=1)
lr_live.fit(X_live_train, y_live_train)

lr_live_train_acc = accuracy_score(y_live_train, lr_live.predict(X_live_train))
lr_live_test_acc  = accuracy_score(y_live_test,  lr_live.predict(X_live_test))

print(f"Logistic Regression -> Train Accuracy: {lr_live_train_acc*100:.2f}%")
print(f"Logistic Regression -> Test Accuracy : {lr_live_test_acc*100:.2f}%")

### 11.2 – Random Forest (Live)

In [ ]:
rf_live = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_live.fit(X_live_train, y_live_train)

rf_live_train_acc = accuracy_score(y_live_train, rf_live.predict(X_live_train))
rf_live_test_acc  = accuracy_score(y_live_test,  rf_live.predict(X_live_test))

print(f"Random Forest -> Train Accuracy: {rf_live_train_acc*100:.2f}%")
print(f"Random Forest -> Test Accuracy : {rf_live_test_acc*100:.2f}%")

### 11.3 – SVM (Live)

In [ ]:
svm_live = SVC(kernel='rbf', probability=True, random_state=42)
svm_live.fit(X_live_train, y_live_train)

svm_live_train_acc = accuracy_score(y_live_train, svm_live.predict(X_live_train))
svm_live_test_acc  = accuracy_score(y_live_test,  svm_live.predict(X_live_test))

print(f"SVM -> Train Accuracy: {svm_live_train_acc*100:.2f}%")
print(f"SVM -> Test Accuracy : {svm_live_test_acc*100:.2f}%")

### 11.4 – Naive Bayes (Live)

In [ ]:
nb_live = GaussianNB()
nb_live.fit(X_live_train, y_live_train)

nb_live_train_acc = accuracy_score(y_live_train, nb_live.predict(X_live_train))
nb_live_test_acc  = accuracy_score(y_live_test,  nb_live.predict(X_live_test))

print(f"Naive Bayes -> Train Accuracy: {nb_live_train_acc*100:.2f}%")
print(f"Naive Bayes -> Test Accuracy : {nb_live_test_acc*100:.2f}%")

## Step 12 – Live Model Comparison & Best Model

In [ ]:
live_results_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'SVM', 'Naive Bayes'],
    'Train Accuracy (%)': [
        round(lr_live_train_acc*100, 2), round(rf_live_train_acc*100, 2),
        round(svm_live_train_acc*100, 2), round(nb_live_train_acc*100, 2)
    ],
    'Test Accuracy (%)': [
        round(lr_live_test_acc*100, 2), round(rf_live_test_acc*100, 2),
        round(svm_live_test_acc*100, 2), round(nb_live_test_acc*100, 2)
    ]
})
live_results_df = live_results_df.sort_values('Test Accuracy (%)', ascending=False).reset_index(drop=True)
print("Live In-Match Model Results:")
live_results_df

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x     = np.arange(len(live_results_df))
width = 0.35

bars1 = ax.bar(x - width/2, live_results_df['Train Accuracy (%)'], width, label='Train Accuracy', color='mediumseagreen', edgecolor='black')
bars2 = ax.bar(x + width/2, live_results_df['Test Accuracy (%)'],  width, label='Test Accuracy',  color='mediumpurple',   edgecolor='black')

ax.set_xlabel('Model')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Live In-Match Model Accuracy Comparison', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(live_results_df['Model'], rotation=15, ha='right')
ax.set_ylim(0, 115)
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f"{bar.get_height():.1f}", ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f"{bar.get_height():.1f}", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
live_model_dict = {
    'Logistic Regression': (lr_live,  lr_live_test_acc),
    'Random Forest':       (rf_live,  rf_live_test_acc),
    'SVM':                 (svm_live, svm_live_test_acc),
    'Naive Bayes':         (nb_live,  nb_live_test_acc)
}

best_live_name  = max(live_model_dict, key=lambda k: live_model_dict[k][1])
best_live_model, best_live_acc = live_model_dict[best_live_name]

print("=" * 50)
print(f"  🏆 Best Live Model  : {best_live_name}")
print(f"  ✅ Test Accuracy    : {best_live_acc*100:.2f}%")
print("=" * 50)

## Step 13 – Confusion Matrix (Best Live Model)

In [ ]:
y_live_pred = best_live_model.predict(X_live_test)
cm = confusion_matrix(y_live_test, y_live_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Batting Team Loses', 'Batting Team Wins'])
disp.plot(colorbar=False, cmap='Greens')
plt.title(f'{best_live_name} – Live Prediction Confusion Matrix')
plt.show()

print(classification_report(y_live_test, y_live_pred, target_names=['Batting Team Loses', 'Batting Team Wins']))

## Step 14 – Feature Importance (Live – Random Forest)

In [ ]:
live_importance = pd.Series(rf_live.feature_importances_, index=live_features).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
live_importance.plot(kind='bar', color='mediumseagreen', edgecolor='black')
plt.title('Feature Importance for Live In-Match Prediction (Random Forest)', fontsize=13)
plt.ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Step 15 – Live In-Match Prediction Function

Input: **batting team, bowling team, runs scored, balls bowled, wickets fallen, target**

Output: **win probability for both teams**

In [ ]:
def predict_live(batting_team, bowling_team, target,
                 current_score, balls_bowled, wickets_fallen,
                 model=best_live_model):

    runs_left    = target - current_score
    balls_left   = 120 - balls_bowled
    wickets_left = 10  - wickets_fallen
    current_over = (balls_bowled // 6) + 1
    crr          = (current_score / balls_bowled) * 6 if balls_bowled > 0 else 0
    rrr          = (runs_left / balls_left) * 6       if balls_left > 0  else 99

    bat_enc  = le_live.transform([batting_team])[0]  if batting_team  in le_live.classes_ else 0
    bowl_enc = le_live.transform([bowling_team])[0]  if bowling_team in le_live.classes_ else 0

    inp = pd.DataFrame([[
        bat_enc, bowl_enc,
        current_score, runs_left, balls_left, wickets_left,
        current_over, target, crr, rrr
    ]], columns=live_features)

    proba  = model.predict_proba(inp)[0]
    over_d = balls_bowled // 6
    ball_d = balls_bowled % 6

    print(f"\n{'='*55}")
    print(f"  🏏 {batting_team} chasing {target}")
    print(f"  vs {bowling_team}")
    print(f"{'='*55}")
    print(f"  Over         : {over_d}.{ball_d}")
    print(f"  Score        : {current_score}/{wickets_fallen}")
    print(f"  Runs needed  : {runs_left} off {balls_left} balls")
    print(f"  Wickets left : {wickets_left}")
    print(f"  CRR          : {crr:.2f}   |   RRR : {rrr:.2f}")
    print(f"{'='*55}")
    print(f"  🟢 {batting_team} win chance  : {proba[1]*100:.1f}%")
    print(f"  🔴 {bowling_team} win chance : {proba[0]*100:.1f}%")
    print(f"{'='*55}")

print("predict_live() ready ✅")

In [ ]:
predict_live(
    batting_team   = 'Mumbai Indians',
    bowling_team   = 'Chennai Super Kings',
    target         = 175,
    current_score  = 80,
    balls_bowled   = 60,
    wickets_fallen = 2
)

In [ ]:
predict_live(
    batting_team   = 'Kolkata Knight Riders',
    bowling_team   = 'Royal Challengers Bangalore',
    target         = 190,
    current_score  = 150,
    balls_bowled   = 96,
    wickets_fallen = 5
)

In [ ]:
predict_live(
    batting_team   = 'Rajasthan Royals',
    bowling_team   = 'Sunrisers Hyderabad',
    target         = 160,
    current_score  = 40,
    balls_bowled   = 30,
    wickets_fallen = 4
)

## Summary

### Part A – Pre-Match Prediction
| Model | Train Accuracy | Test Accuracy |
|---|---|---|
| Logistic Regression | ~73% | ~72% |
| Random Forest | ~97% | ~65% |
| SVM | ~75% | ~68% |
| Naive Bayes | ~62% | ~60% |

### Part B – Live In-Match Prediction (Runs / Balls / Wickets / Overs)
| Model | Train Accuracy | Test Accuracy |
|---|---|---|
| Logistic Regression | ~77% | ~77% |
| Random Forest | ~88% | ~81% |
| SVM | ~80% | ~79% |
| Naive Bayes | ~72% | ~72% |

**Key Insight:** Live in-match prediction is significantly more accurate because runs, balls, wickets and required run rate are strong real-time signals of match outcome.